# PhasePhyto Apple Overlap Colab Pipeline

This notebook downloads the three apple-overlap source datasets into Google Drive, stores them as **tar archives** to save memory and speed up future sessions, prepares the strict shared-label overlap subset, archives that overlap subset too, then hydrates the overlap archive onto Colab SSD for training/evaluation.

Shared labels:

- `Apple___healthy`
- `Apple___Apple_scab`
- `Apple___Cedar_apple_rust`

Datasets used:

- PlantVillage
- PlantDoc
- Plant Pathology 2021


In [ ]:
# ============================================================
# CONFIGURATION -- edit only if you want different paths/behavior
# ============================================================

CONFIG = {
    "drive_project_dir": "/content/drive/MyDrive/PhasePhyto",
    "repo_url": "https://github.com/kautilyaa/PhasePhyto.git",
    "repo_branch": "aws_changes",
    "repo_dir": "/content/PhasePhyto",
    "kaggle_json_drive_path": "/content/drive/MyDrive/kaggle.json",
    "download_plantvillage": True,
    "download_plantdoc": True,
    "download_plant_pathology_2021": True,
    "force_redownload": False,
    "recreate_archives": False,
    "rebuild_overlap": True,
    "overlap_mode": "copy",  # use copy so the overlap archive is self-contained
    "hydrate_overlap_to_ssd": True,
    "run_train": False,
    "run_eval_plantdoc": False,
    "run_eval_pp2021": False,
    "checkpoint_dir": "/content/drive/MyDrive/PhasePhyto/checkpoints/apple_overlap_plantdoc",
}

for k, v in CONFIG.items():
    print(f"{k}: {v}")


In [ ]:
# ============================================================
# MOUNT DRIVE + CLONE/INSTALL REPO + DEFINE PATHS
# ============================================================

from pathlib import Path
import os
import shutil
import subprocess
import sys
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

REPO_DIR = Path(CONFIG["repo_dir"])
if not REPO_DIR.exists():
    subprocess.run([
        "git", "clone", "--branch", CONFIG["repo_branch"],
        CONFIG["repo_url"], str(REPO_DIR)
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--quiet"], check=False)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", CONFIG["repo_branch"], "--quiet"], check=False)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--quiet"], check=False)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

DRIVE_PROJECT_DIR = Path(CONFIG["drive_project_dir"])
PLANT_DISEASE_ROOT = DRIVE_PROJECT_DIR / "data" / "plant_disease"
BENCHMARK_ROOT = DRIVE_PROJECT_DIR / "data" / "plant_benchmarks"
ARCHIVE_DIR = DRIVE_PROJECT_DIR / "data" / "archives"
OVERLAP_ROOT_DRIVE = DRIVE_PROJECT_DIR / "data" / "overlap" / "apple_strict"
OVERLAP_ARCHIVE = ARCHIVE_DIR / "apple_strict.tar"
LOCAL_DATA_ROOT = Path("/content/data")
LOCAL_OVERLAP_PARENT = LOCAL_DATA_ROOT / "overlap"
LOCAL_OVERLAP_ROOT = LOCAL_OVERLAP_PARENT / "apple_strict"

PLANTVILLAGE_DIR = PLANT_DISEASE_ROOT / "plantvillage"
PLANTDOC_DIR = PLANT_DISEASE_ROOT / "plantdoc"
PP2021_DIR = BENCHMARK_ROOT / "plant_pathology_2021"

for path in [DRIVE_PROJECT_DIR, PLANT_DISEASE_ROOT, BENCHMARK_ROOT, ARCHIVE_DIR, OVERLAP_ROOT_DRIVE.parent, LOCAL_DATA_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print("Repo ready:", REPO_DIR)
print("Plant disease root:", PLANT_DISEASE_ROOT)
print("Benchmark root:", BENCHMARK_ROOT)
print("Archive dir:", ARCHIVE_DIR)
print("Overlap root (Drive):", OVERLAP_ROOT_DRIVE)


In [ ]:
# ============================================================
# HELPERS + KAGGLE CREDENTIALS
# ============================================================

import tarfile


def run(cmd):
    print("\n$ " + " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), check=True)


def ensure_kaggle_credentials() -> None:
    kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
    kaggle_json.parent.mkdir(parents=True, exist_ok=True)
    drive_copy = Path(CONFIG["kaggle_json_drive_path"])

    if kaggle_json.exists():
        os.chmod(kaggle_json, 0o600)
        print("Kaggle credentials already exist:", kaggle_json)
        return

    if drive_copy.exists():
        shutil.copy2(drive_copy, kaggle_json)
        os.chmod(kaggle_json, 0o600)
        print("Copied Kaggle credentials from Drive:", drive_copy)
        return

    print("Upload kaggle.json now (Kaggle > Account > Create New API Token).")
    from google.colab import files
    uploaded = files.upload()
    if "kaggle.json" not in uploaded:
        raise RuntimeError("kaggle.json was not uploaded.")
    shutil.move("kaggle.json", kaggle_json)
    shutil.copy2(kaggle_json, drive_copy)
    os.chmod(kaggle_json, 0o600)
    print("Saved reusable kaggle.json to:", drive_copy)


def create_or_refresh_tar(source_dir: Path, archive_path: Path, *, force: bool) -> None:
    if not source_dir.exists():
        raise FileNotFoundError(f"Source directory missing: {source_dir}")
    if archive_path.exists() and not force:
        print(f"Archive already exists: {archive_path}")
        return
    archive_path.parent.mkdir(parents=True, exist_ok=True)
    if archive_path.exists():
        archive_path.unlink()
    with tarfile.open(archive_path, "w") as tf:
        tf.add(source_dir, arcname=source_dir.name)
    size_gb = archive_path.stat().st_size / 1e9
    print(f"Created archive: {archive_path} ({size_gb:.2f} GB)")


ensure_kaggle_credentials()


In [ ]:
# ============================================================
# DOWNLOAD RAW DATASETS TO DRIVE (only once; skipped if already present)
# ============================================================

if CONFIG["download_plantvillage"]:
    run([
        sys.executable, "scripts/download_data.py",
        "--dataset", "plantvillage",
        "--output", str(PLANT_DISEASE_ROOT),
    ])

if CONFIG["download_plantdoc"]:
    run([
        sys.executable, "scripts/download_data.py",
        "--dataset", "plantdoc",
        "--output", str(PLANT_DISEASE_ROOT),
    ])

if CONFIG["download_plant_pathology_2021"]:
    run([
        sys.executable, "scripts/download_data.py",
        "--dataset", "plant_pathology_2021",
        "--output", str(BENCHMARK_ROOT),
    ])


In [ ]:
# ============================================================
# CREATE/REFRESH DRIVE TAR ARCHIVES FOR RAW DATASETS
# ============================================================

create_or_refresh_tar(
    PLANTVILLAGE_DIR,
    ARCHIVE_DIR / "plantvillage.tar",
    force=CONFIG["recreate_archives"],
)
create_or_refresh_tar(
    PLANTDOC_DIR,
    ARCHIVE_DIR / "plantdoc.tar",
    force=CONFIG["recreate_archives"],
)
create_or_refresh_tar(
    PP2021_DIR,
    ARCHIVE_DIR / "plant_pathology_2021.tar",
    force=CONFIG["recreate_archives"],
)


In [ ]:
# ============================================================
# BUILD STRICT APPLE OVERLAP ON DRIVE + ARCHIVE IT
# ============================================================

if CONFIG["rebuild_overlap"] and OVERLAP_ROOT_DRIVE.exists():
    shutil.rmtree(OVERLAP_ROOT_DRIVE)

run([
    sys.executable, "scripts/prepare_overlap_datasets.py",
    "--plantvillage", str(PLANTVILLAGE_DIR),
    "--plantdoc", str(PLANTDOC_DIR),
    "--plant-pathology-2021", str(PP2021_DIR),
    "--output", str(OVERLAP_ROOT_DRIVE),
    "--mode", CONFIG["overlap_mode"],
    "--clean",
])

create_or_refresh_tar(
    OVERLAP_ROOT_DRIVE,
    OVERLAP_ARCHIVE,
    force=CONFIG["recreate_archives"] or CONFIG["rebuild_overlap"],
)


In [ ]:
# ============================================================
# HYDRATE OVERLAP TAR TO COLAB SSD (/content/data) FOR TRAINING SPEED
# ============================================================

if CONFIG["hydrate_overlap_to_ssd"]:
    if LOCAL_OVERLAP_PARENT.exists():
        shutil.rmtree(LOCAL_OVERLAP_PARENT)
    LOCAL_OVERLAP_PARENT.mkdir(parents=True, exist_ok=True)
    with tarfile.open(OVERLAP_ARCHIVE) as tf:
        tf.extractall(LOCAL_OVERLAP_PARENT)
    print("Hydrated overlap archive to SSD:", LOCAL_OVERLAP_ROOT)
else:
    print("Skipped overlap hydration; use Drive paths directly if you want.")


In [ ]:
# ============================================================
# SANITY CHECK COUNTS
# ============================================================

from phasephyto.data.class_mapping import APPLE_STRICT_CLASSES

for dataset_name in ["plantvillage", "plantdoc", "plant_pathology_2021"]:
    ds_root = LOCAL_OVERLAP_ROOT / dataset_name
    print(f"\nDATASET: {dataset_name}")
    for class_name in APPLE_STRICT_CLASSES:
        class_dir = ds_root / class_name
        n = len([p for p in class_dir.glob("*") if p.is_file()]) if class_dir.exists() else 0
        print(f"  {class_name}: {n}")


In [ ]:
# ============================================================
# TRAIN: PlantVillage source -> PlantDoc target overlap benchmark
# ============================================================

if CONFIG["run_train"]:
    run([
        sys.executable, "-m", "phasephyto.train",
        "--config", "configs/apple_overlap_plantdoc.yaml",
        "--override",
        f"checkpoint_dir={CONFIG['checkpoint_dir']}",
        f"data.root={LOCAL_OVERLAP_ROOT}",
        f"data.source_dir={LOCAL_OVERLAP_ROOT / 'plantvillage'}",
        f"data.target_dir={LOCAL_OVERLAP_ROOT / 'plantdoc'}",
        f"data.eval_source_dir={LOCAL_OVERLAP_ROOT / 'plantvillage'}",
        f"data.eval_target_dir={LOCAL_OVERLAP_ROOT / 'plantdoc'}",
    ])
else:
    print("Set CONFIG['run_train'] = True to launch training from this notebook.")


In [ ]:
# ============================================================
# EVAL 1: PlantVillage -> PlantDoc overlap
# ============================================================

if CONFIG["run_eval_plantdoc"]:
    run([
        sys.executable, "-m", "phasephyto.evaluate",
        "--config", "configs/apple_overlap_plantdoc.yaml",
        "--checkpoint", str(Path(CONFIG["checkpoint_dir"]) / "best_model.pt"),
        "--source-dir", str(LOCAL_OVERLAP_ROOT / "plantvillage"),
        "--target-dir", str(LOCAL_OVERLAP_ROOT / "plantdoc"),
        "--output", str(Path(CONFIG["checkpoint_dir"]) / "eval_plantdoc.json"),
    ])
else:
    print("Set CONFIG['run_eval_plantdoc'] = True to run PlantDoc evaluation.")


In [ ]:
# ============================================================
# EVAL 2: PlantVillage -> Plant Pathology 2021 overlap
# ============================================================

if CONFIG["run_eval_pp2021"]:
    run([
        sys.executable, "-m", "phasephyto.evaluate",
        "--config", "configs/apple_overlap_pp2021.yaml",
        "--checkpoint", str(Path(CONFIG["checkpoint_dir"]) / "best_model.pt"),
        "--source-dir", str(LOCAL_OVERLAP_ROOT / "plantvillage"),
        "--target-dir", str(LOCAL_OVERLAP_ROOT / "plant_pathology_2021"),
        "--output", str(Path(CONFIG["checkpoint_dir"]) / "eval_pp2021.json"),
    ])
else:
    print("Set CONFIG['run_eval_pp2021'] = True to run Plant Pathology 2021 evaluation.")


## Optional next step: batch inference notebook

After you have the four ablation checkpoints, open:

- `notebooks/PhasePhyto_Batch_Inference.ipynb`

and configure `DATASET_RUNS` with:

- `/content/data/overlap/apple_strict/plantdoc`
- `/content/data/overlap/apple_strict/plant_pathology_2021`

using `/content/data/overlap/apple_strict/plantvillage` as `class_to_idx_source`.
